# Embeddings & Vector Databases
Convert text to dense vectors. Search by meaning, not keywords.

In [ ]:
import micropip
await micropip.install(['scikit-learn','matplotlib','numpy'])
print('Ready!')

## What are Embeddings?

'king' - 'man' + 'woman' = 'queen' (Word2Vec)

Modern sentence embeddings: one 384-1536 dim vector per sentence capturing full meaning.
Similar sentences have high cosine similarity.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
np.random.seed(42)
n = 5
clusters = {
    'ML':    np.random.randn(n,50) + np.array([3,0]+[0]*48),
    'DL':    np.random.randn(n,50) + np.array([0,3]+[0]*48),
    'GenAI': np.random.randn(n,50) + np.array([-3,0]+[0]*48),
    'Prog':  np.random.randn(n,50) + np.array([0,-3]+[0]*48),
}
X = np.vstack(list(clusters.values()))
labels = [lb for lb,pts in clusters.items() for _ in range(n)]
X2d = PCA(n_components=2).fit_transform(X)
colors = {'ML':'#6b21a8','DL':'#059669','GenAI':'#f59e0b','Prog':'#2563eb'}
plt.figure(figsize=(7,6))
for i,(x,y) in enumerate(X2d):
    plt.scatter(x,y,c=colors[labels[i]],s=80)
    plt.annotate(labels[i],(x,y),fontsize=7,ha='center',va='bottom')
plt.title('Sentence Embeddings in 2D -- similar topics cluster')
plt.grid(alpha=0.3); plt.show()

In [ ]:
# Cosine similarity
def cosim(a,b): return np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b))
vecs = {'king':np.array([0.9,0.1,0.8]),'queen':np.array([0.8,0.9,0.8]),
        'man':np.array([0.9,0.1,0.1]),'woman':np.array([0.8,0.9,0.1]),
        'computer':np.array([0.0,0.0,0.9]),'laptop':np.array([0.1,0.0,0.85])}
print('Cosine similarities:')
for a,b in [('king','queen'),('king','man'),('man','woman'),('computer','laptop'),('king','computer')]:
    s = cosim(vecs[a],vecs[b])
    print(f'  {a:<10} <-> {b:<10}: {s:.3f} {"#"*int(s*25)}')

In [ ]:
# Semantic search
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
docs = [
    'Linear regression predicts continuous values by fitting a line',
    'Random forest combines decision trees to reduce variance',
    'Neural networks stack layers with activation functions',
    'BERT learns bidirectional context via masked language modelling',
    'K-Means clustering assigns points to nearest centroid',
    'Gradient descent minimises loss by moving along negative gradient',
    'CNNs detect spatial features in images using convolutional filters',
    'Transformers use self-attention to model token relationships',
    'RAG grounds LLM responses in retrieved documents',
]
vec = TfidfVectorizer(stop_words='english')
V = vec.fit_transform(docs)
def search(q,k=3):
    scores = cosine_similarity(vec.transform([q]),V).flatten()
    top = np.argsort(scores)[-k:][::-1]
    print(f'Query: "{q}"')
    for i in top:
        print(f'  [{scores[i]:.3f}] {docs[i]}')
    print()
search('how do trees work in machine learning')
search('language model pretraining objectives')

## Production Vector DBs

```python
import chromadb
from sentence_transformers import SentenceTransformer  # free, local

embedder = SentenceTransformer('all-MiniLM-L6-v2')
client = chromadb.Client()
coll = client.create_collection('docs')

for i, doc in enumerate(docs):
    coll.add(ids=[str(i)], embeddings=[embedder.encode(doc).tolist()], documents=[doc])

results = coll.query(query_embeddings=[embedder.encode('deep learning images').tolist()], n_results=3)
print(results['documents'][0])
```

| DB | Best for | Scale |
|---|---|---|
| ChromaDB | Prototyping, local | Small-medium |
| Pinecone | Production SaaS | Large |
| Supabase pgvector | Already on Supabase | Medium |

---
**Your turn:** Add 5 GenAI docs to the search index. Do queries about 'transformers' find them correctly?